In [ ]:
from datasets import load_dataset

DATASET_ID = "jg-eno/ml-gsm8k-test"

ml_ds = load_dataset(DATASET_ID, split="test")

print("Number of examples:", len(ml_ds))
print("Columns:", ml_ds.column_names)
print("\nFirst example:")
print(ml_ds[0])

Number of examples: 840
Columns: ['id', 'question_en', 'question_ml', 'answer_en', 'answer_ml']

First example:
{'id': 'train_0', 'question_en': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'question_ml': 'നതാലിയ ഏപ്രിലിൽ അവളുടെ 48 സുഹൃത്തുക്കൾക്ക് ക്ലിപ്പുകൾ വിറ്റു, തുടർന്ന് മെയ് മാസത്തിൽ അവൾ പകുതി ക്ലിപ്പുകൾ വിറ്റു. ഏപ്രിലിലും മെയ് മാസത്തിലുമായി നതാലിയ ആകെ എത്ര ക്ലിപ്പുകൾ വിറ്റു?', 'answer_en': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'answer_ml': 'നതാലിയ മെയ് മാസത്തിൽ 48/2 = <<48/2=24>>24 ക്ലിപ്പുകൾ വിറ്റു.\nനതാലിയ ഏപ്രിൽ, മെയ് മാസങ്ങളിൽ ആകെ 48+24 = <<48+24=72>>72 ക്ലിപ്പുകൾ വിറ്റു.\n#### 72'}


In [ ]:
dataset_dict = load_dataset(DATASET_ID)
print(dataset_dict)

DatasetDict({
    test: Dataset({
        features: ['id', 'question_en', 'question_ml', 'answer_en', 'answer_ml'],
        num_rows: 840
    })
})


In [ ]:
from datasets import load_dataset
import random
import re
from decimal import Decimal, InvalidOperation

DATASET_ID = "jg-eno/ml-gsm8k-test"
SEED = 42
N_EVAL = 500

ml_ds = load_dataset(DATASET_ID, split="test")

print("Dataset size:", len(ml_ds))
print("Fields:", ml_ds.column_names)

def normalize_number(value):
    if value is None:
        return None

    value = str(value).strip().replace(",", "")

    try:
        number = Decimal(value)
        if not number.is_finite():
            return None
        return format(number.normalize(), "f")
    except InvalidOperation:
        return None

def extract_gold_answer(text):
    matches = re.findall(
        r"####\s*<?\s*(-?\d[\d,]*(?:\.\d+)?)\s*>?",
        str(text),
    )
    return normalize_number(matches[-1]) if matches else None

valid_rows = []

for idx, row in enumerate(ml_ds):
    question = str(row["question_ml"] or "").strip()
    gold = extract_gold_answer(row["answer_en"])

    if question and gold is not None:
        valid_rows.append({
            "dataset_index": idx,
            "id": row["id"],
            "question": question,
            "gold": gold,
        })

rng = random.Random(SEED)
rng.shuffle(valid_rows)
eval_rows = valid_rows[:min(N_EVAL, len(valid_rows))]

assert eval_rows, "No valid evaluation examples found."

print("Valid examples:", len(valid_rows))
print("Selected examples:", len(eval_rows))
print("Sample question:", eval_rows[0]["question"])
print("Gold answer:", eval_rows[0]["gold"])

Dataset size: 840
Fields: ['id', 'question_en', 'question_ml', 'answer_en', 'answer_ml']
Valid examples: 840
Selected examples: 500
Sample question: കൊക്കോയുടെ പട്ടണത്തിലെ വൈദ്യുതി വില ഒരു kW-ന് $0.10 ആണ്. കൊക്കോയുടെ പുതിയ ഓവൻ 2.4 kWh (കിലോവാട്ട്-മണിക്കൂർ) ആണ് ഉപയോഗിക്കുന്ന നിരക്ക്. കഴിഞ്ഞ മാസം ആകെ 25 മണിക്കൂർ ഓവൻ ഉപയോഗിച്ചാൽ കൊക്കോ എത്ര രൂപ നൽകണം?
Gold answer: 6


In [ ]:
for i in range(5):
    row = eval_rows[i]
    print(f"\n--- Example {i + 1} | ID: {row['id']} ---")
    print("Malayalam:", row["question"])
    print("English reference:", ml_ds[row["dataset_index"]]["question_en"])
    print("Gold:", row["gold"])


--- Example 1 | ID: train_509 ---
Malayalam: കൊക്കോയുടെ പട്ടണത്തിലെ വൈദ്യുതി വില ഒരു kW-ന് $0.10 ആണ്. കൊക്കോയുടെ പുതിയ ഓവൻ 2.4 kWh (കിലോവാട്ട്-മണിക്കൂർ) ആണ് ഉപയോഗിക്കുന്ന നിരക്ക്. കഴിഞ്ഞ മാസം ആകെ 25 മണിക്കൂർ ഓവൻ ഉപയോഗിച്ചാൽ കൊക്കോ എത്ര രൂപ നൽകണം?
English reference: The electricity price in Coco's town is $0.10 per kW. Coco's new oven has a consumption rate of 2.4 kWh (kilowatt-hours). How much will Coco pay for using his oven only if he used it for a total of 25 hours last month?
Gold: 6

--- Example 2 | ID: train_368 ---
Malayalam: ലിൻഡ രണ്ട് കളറിംഗ് പുസ്തകങ്ങൾ 4 ഡോളർ വീതം, 4 പാക്കറ്റ് നിലക്കടല 1.50 ഡോളർ വീതം, ഒരു സ്റ്റഫ്ഡ് മൃഗം എന്നിവ വാങ്ങി. അവൾ കാഷ്യർക്ക് 25 ഡോളർ കൊടുത്തു, ബാക്കി തിരികെ കിട്ടിയില്ല. ഒരു സ്റ്റഫ്ഡ് മൃഗത്തിന് എത്രയാണ് വില?
English reference: Linda bought two coloring books at $4 each, 4 packs of peanuts at $1.50 each pack, and one stuffed animal. She gave the cashier $25 and got no change. How much does a stuffed animal cost?
Gold: 11

--- Example 3 | ID: train_638 -

In [ ]:
import torch
import re
import json
import random
from pathlib import Path
from decimal import Decimal, InvalidOperation
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForMultimodalLM

# ============================================================
# 1. Configuration
# ============================================================

MODEL_ID = "Qwen/Qwen3.5-2B"
DATASET_ID = "jg-eno/ml-gsm8k-test"

SEED = 42
N_EVAL = 500
BATCH_SIZE = 4
MAX_NEW_TOKENS = 768

OUTPUT_DIR = Path("/content/malayalam_gsm8k_eval")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_PATH = OUTPUT_DIR / "predictions.jsonl"
METRICS_PATH = OUTPUT_DIR / "metrics.json"

torch.manual_seed(SEED)
random.seed(SEED)

# ============================================================
# 2. Load and prepare evaluation dataset
# ============================================================

ml_ds = load_dataset(DATASET_ID, split="test")

def normalize_number(value):
    if value is None:
        return None

    value = str(value).strip().replace(",", "")

    try:
        number = Decimal(value)
        if not number.is_finite():
            return None
        return format(number.normalize(), "f")
    except (InvalidOperation, ValueError):
        return None

def extract_gold_answer(text):
    matches = re.findall(
        r"####\s*<?\s*(-?\d[\d,]*(?:\.\d+)?)\s*>?",
        str(text),
    )
    return normalize_number(matches[-1]) if matches else None

valid_rows = []

for idx, row in enumerate(ml_ds):
    question = str(row["question_ml"] or "").strip()
    gold = extract_gold_answer(row["answer_en"])

    if question and gold is not None:
        valid_rows.append({
            "dataset_index": idx,
            "id": row["id"],
            "question": question,
            "gold": gold,
        })

rng = random.Random(SEED)
rng.shuffle(valid_rows)
eval_rows = valid_rows[:min(N_EVAL, len(valid_rows))]

assert len(eval_rows) > 0, "No valid evaluation examples."
assert len(eval_rows) == min(N_EVAL, len(valid_rows))

print(f"Dataset size: {len(ml_ds)}")
print(f"Valid examples: {len(valid_rows)}")
print(f"Evaluating: {len(eval_rows)}")

# ============================================================
# 3. Load original Qwen3.5-2B
# ============================================================

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=(
        torch.bfloat16
        if torch.cuda.is_available()
        and torch.cuda.is_bf16_supported()
        else torch.float16
    ),
    device_map="auto",
).eval()

tokenizer = processor.tokenizer

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"
model.config.use_cache = True

print("Model loaded:", MODEL_ID)
print("Device:", model.device)
print("Tokenizer vocabulary size:", len(tokenizer))

# ============================================================
# 4. Prompt construction
# ============================================================

SYSTEM_PROMPT = (
    "നിങ്ങൾ ഒരു ഗണിത പ്രശ്നപരിഹാര സഹായിയാണ്. "
    "ചോദ്യത്തിന് ഘട്ടംഘട്ടമായി മലയാളത്തിൽ ഉത്തരം നൽകുക. "
    "അവസാന വരിയിൽ #### എന്നതിന് ശേഷം അന്തിമ സംഖ്യാ ഉത്തരം നൽകുക. "
    "ഉദാഹരണത്തിന്: #### 42"
)

def make_prompt(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    return processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

# ============================================================
# 5. Prediction extraction
# ============================================================

def extract_prediction(text):
    patterns = [
        r"####\s*<?\s*(-?\d[\d,]*(?:\.\d+)?)\s*>?",
        r"\\boxed\{\s*(-?\d[\d,]*(?:\.\d+)?)\s*\}",
        r"(?:final answer|the answer is|answer)"
        r"\s*[:：]?\s*(-?\d[\d,]*(?:\.\d+)?)",
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        if matches:
            return normalize_number(matches[-1])

    return None

# ============================================================
# 6. Batched inference
# ============================================================

results = []
model.eval()

for start in tqdm(
    range(0, len(eval_rows), BATCH_SIZE),
    desc="Evaluating Qwen3.5-2B",
):
    batch = eval_rows[start:start + BATCH_SIZE]
    prompts = [make_prompt(row["question"]) for row in batch]

    inputs = processor(
        text=prompts,
        padding=True,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(model.device) if torch.is_tensor(value) else value
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_length = inputs["input_ids"].shape[1]
    generated_ids = output_ids[:, prompt_length:]

    decoded = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )

    for row, output in zip(batch, decoded):
        predicted_answer = extract_prediction(output)
        gold = row["gold"]

        results.append({
            "dataset_index": row["dataset_index"],
            "id": row["id"],
            "question": row["question"],
            "gold_answer": gold,
            "prediction": output,
            "predicted_answer": predicted_answer,
            "correct": (
                predicted_answer is not None
                and predicted_answer == gold
            ),
        })

# ============================================================
# 7. Compute benchmark statistics
# ============================================================

total = len(results)
correct = sum(result["correct"] for result in results)
extracted = sum(
    result["predicted_answer"] is not None
    for result in results
)

unparseable = total - extracted
incorrect_extracted = extracted - correct

metrics = {
    "Evaluated examples": total,
    "Correct answers": correct,
    "Numeric exact-match accuracy (%)": round(
        100 * correct / total, 2
    ) if total else 0.0,
    "Answer extraction coverage (%)": round(
        100 * extracted / total, 2
    ) if total else 0.0,
    "Accuracy among extracted answers (%)": round(
        100 * correct / extracted, 2
    ) if extracted else 0.0,
    "Unparseable predictions": unparseable,
    "Incorrect extracted answers": incorrect_extracted,
}

# ============================================================
# 8. Save predictions and metrics
# ============================================================

with PREDICTIONS_PATH.open("w", encoding="utf-8") as f:
    for result in results:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

with METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 55)
print("MALAYALAM GSM8K EVALUATION RESULTS")
print("=" * 55)

print(json.dumps(metrics, indent=2, ensure_ascii=False))

print("\nSaved files:")
print("Predictions:", PREDICTIONS_PATH)
print("Metrics:", METRICS_PATH)

# ============================================================
# 9. Inspect sample errors
# ============================================================

print("\nSample incorrect predictions:")

errors = [r for r in results if not r["correct"]]

for result in errors[:5]:
    print("\nID:", result["id"])
    print("Gold:", result["gold_answer"])
    print

Dataset size: 840
Valid examples: 840
Evaluating: 500


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

...

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen3.5-2B
Device: cuda:0
Tokenizer vocabulary size: 248077


Evaluating Qwen3.5-2B:   0%|          | 0/125 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
